In [2]:
# ============================================================
# CP Intonation Visualizer — Batch + Interactive + Presets + Clean Titles
# UI revamp + LOWESS vs GAM (statsmodels) + Auto Flex for GAM
# Auto Flex tuned for expressive contours (df~50, λ~0.005 at high flex)
# ============================================================

import os
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.interpolate import make_interp_spline
from statsmodels.nonparametric.smoothers_lowess import lowess
import parselmouth
from praatio import textgrid

# Try to enable GAM (statsmodels.gam); fall back gracefully if unavailable
_HAVE_GAM = True
try:
    import statsmodels as _sm
    import statsmodels.api as sm
    from statsmodels.gam.api import GLMGam, BSplines
    _SM_VER = getattr(_sm, "__version__", "unknown")
except Exception:
    _HAVE_GAM = False
    _SM_VER = "not-installed"

# Debug flag: set True to see fallback reasons in notebook output
_GAM_DEBUG = False

# --- UI (Jupyter) ---
try:
    from IPython.display import display, clear_output, HTML
    import ipywidgets as W
    _HAVE_WIDGETS = True
except Exception:
    _HAVE_WIDGETS = False

# === Phonetic sets (SAMPA-like)
vocalic_segments = {"a", "A", "e", "E", "i", "I", "o", "O", "u", "U", "j", "w", "@"}
voiced_consonants = {"b", "d", "g", "l", "L", "m", "n", "N", "J", "z", "Z", "v"}
target_phones = vocalic_segments.union(voiced_consonants)

# === Presets: suggest f0 range + smoothing for voice categories
PRESETS = {
    "Custom":     {"fmin": None, "fmax": None, "frac": None},
    "Male":       {"fmin": 75,  "fmax": 220, "frac": 0.28},
    "Female":     {"fmin": 120, "fmax": 380, "frac": 0.30},
    "Old":        {"fmin": 85,  "fmax": 300, "frac": 0.34},
}

# ---------- Helpers ----------
def _tier_exists(tg, name):
    """Return True if the TextGrid contains a tier with the given name."""
    return name in tg.tierNames

def is_pause_mark(mark: str) -> bool:
    """
    Return True if a CP/segment label represents a pause or noise.
    Empty labels, <p...> and <RUMORE> are treated as pauses/noise.
    """
    if mark is None:
        return True
    s = str(mark).strip()
    return (s == "" or s.startswith("<p") or s.startswith("<RUMORE>"))

def get_cp_intervals(tg):
    """
    Return all CP intervals (including pauses/noise) as (start, end, label).
    """
    if not _tier_exists(tg, "CP"):
        print("Tier 'CP' not found.")
        return []
    cp_tier = tg.getTier("CP")
    return [(start, end, label) for start, end, label in cp_tier.entries]

def get_valid_cp_intervals(tg):
    """
    Return only 'real' CP intervals, excluding pauses/noise,
    preserving the original sequential order.
    """
    all_cp = get_cp_intervals(tg)
    valid = [(s, e, lab) for (s, e, lab) in all_cp if not is_pause_mark(lab)]
    return valid

def get_voiced_segments(tg, xmin, xmax):
    """
    Return all phonetic intervals within [xmin, xmax] belonging to the 'phon'
    tier that correspond to vowels and voiced consonants in SAMPA-like labels.
    """
    if not _tier_exists(tg, "phon"):
        return []
    phon_tier = tg.getTier("phon")
    return [
        (start, end)
        for start, end, label in phon_tier.entries
        if start >= xmin and end <= xmax and label in target_phones
    ]

def extract_f0_intensity(wav_path, voiced_segments):
    """
    Extract f0 (Hz) and intensity (dB) at 5 ms steps over voiced segments.

    Returns:
        times (np.ndarray): time stamps in seconds
        f0_vals (np.ndarray): fundamental frequency values
        intens_vals (np.ndarray): intensity values (dB)
    """
    snd = parselmouth.Sound(wav_path)
    pitch = snd.to_pitch()
    intensity = snd.to_intensity()

    times, f0_vals, intens_vals = [], [], []
    for start, end in voiced_segments:
        # Skip extremely short segments where estimates are unreliable
        if end - start < 0.02:
            continue
        t_values = np.arange(start, end, 0.005)  # 5 ms step
        for t in t_values:
            f0 = pitch.get_value_at_time(t)
            inten = intensity.get_value(t)
            if f0 and f0 > 0:
                times.append(t)
                f0_vals.append(f0)
                intens_vals.append(float(inten) if inten else 0.0)
    return np.array(times), np.array(f0_vals), np.array(intens_vals)

def filter_f0_outliers(times, f0_values, intens_vals, z_threshold=2.5):
    """
    Lightly filter pitch outliers using a z-score threshold.

    Higher z_threshold (e.g. 2.5) keeps more natural excursions
    so the contour does not become artificially flattened.
    """
    f0_arr = np.array(f0_values, dtype=float)
    if f0_arr.size == 0:
        return times, f0_arr, intens_vals

    mean = np.mean(f0_arr)
    std = np.std(f0_arr)
    if std == 0 or np.isnan(std):
        mask = np.ones_like(f0_arr, dtype=bool)
    else:
        mask = np.abs(f0_arr - mean) < z_threshold * std
    return times[mask], f0_arr[mask], intens_vals[mask]

def syllabic_rate_per_syllable(syll_intervals):
    """
    Compute syllabic rate (SR) per syllable as 1/duration.

    Returns:
        centers (np.ndarray): midpoints of each syllable (s)
        sr_values (np.ndarray): syllabic rate values (syllables/second)
    """
    centers = []
    sr_values = []
    for start, end, lbl in syll_intervals:
        if not lbl.startswith("<p") and not lbl.startswith("<RUMORE>"):
            duration = end - start
            if duration > 0:
                centers.append((start + end) / 2)
                sr_values.append(1.0 / duration)
    return np.array(centers), np.array(sr_values)

def get_current_verse_label(tg, xmin, xmax):
    """
    Retrieve the verse text (VS tier label) covering the given window [xmin, xmax].
    If no VS interval fully spans the window, return an empty string.
    """
    if not _tier_exists(tg, "VS"):
        return ""
    vs_tier = tg.getTier("VS")
    for start, end, label in vs_tier.entries:
        if start <= xmin and end >= xmax:
            return label.strip()
    return ""

def _syllables_in_window(syll_intervals, xmin, xmax):
    """
    Return all syllable intervals fully contained in [xmin, xmax],
    excluding pauses and noise.
    """
    return [
        (s_start, s_end, s_lbl)
        for (s_start, s_end, s_lbl) in syll_intervals
        if (s_start >= xmin and s_end <= xmax
            and not s_lbl.startswith("<p")
            and not s_lbl.startswith("<RUMORE>"))
    ]

def _syllable_metrics(syll_intervals, xmin, xmax):
    """
    Compute basic syllable metrics within [xmin, xmax]:

    Returns:
        n (int): number of syllables
        mean_ms (float): mean syllable duration (ms)
        sr (float): syllabic rate (syllables/second)
    """
    kept = _syllables_in_window(syll_intervals, xmin, xmax)
    n = len(kept)
    durations = [(e - s) for s, e, _ in kept]
    mean_ms = (np.mean(durations) * 1000.0) if durations else np.nan
    cp_dur = max(xmax - xmin, 1e-9)
    sr = n / cp_dur
    return n, mean_ms, sr

# Robust CP title builder with de-duplication
def build_cp_title(cp_seq_num: int, cp_label: str, mode: str = "auto") -> str:
    """
    Build a CP title string while avoiding duplicated labels.

    mode:
        'auto'  → CP number + label only if label adds information
        'number'→ CP number only
        'label' → label only (fallback to CP number if empty)
        'both'  → CP number + label always
    """
    num_tag = f"CP{cp_seq_num}"
    raw = (cp_label or "").strip()

    if mode == "number":
        return num_tag
    if mode == "label":
        return raw if raw else num_tag

    lab_norm = re.sub(r"\s+", "", raw).upper()
    num_norm = num_tag.upper()

    if mode in ("auto", "both"):
        if lab_norm in ("", "CP", num_norm):
            return num_tag
        stripped = re.sub(
            rf"^\s*CP\s*0*{cp_seq_num}\s*[:\-–]?\s*", "", raw, flags=re.IGNORECASE
        ).strip()
        if mode == "both":
            return f"{num_tag}: //{stripped or raw}//"
        else:
            return f"{num_tag}: //{(stripped or raw)}//"

    return f"{num_tag}: //{raw}//"


# ---------- Adaptive GAM Flex ----------
def _gam_params_auto(n_syll: int, flex: int):
    """
    Map syllable count + Flex(0..120) to (df, lambda) tuned for expressive F0.

    Goals:
    - High flex (~120) -> df ≈ 50, λ ≈ 0.005
      (highly detailed curve; preserves local accents)
    - Low flex (~0)    -> smoother global trend
      (less local accent detail)

    Strategy:
    - df comes from syllable count + flex scaling, then capped around 50.
    - λ drops sharply with flex, down to ~0.005.
    """
    # Clamp flex to [0,120]
    f = max(0, min(int(flex), 120))

    # Baseline df from syllable count.
    # Longer CP => more meaningful contour events => allow more bends.
    n = max(1, int(n_syll))
    df_syll = 2 * n + 8  # e.g., n=10 -> 28

    # Scale df with flex.
    # flex = 0   -> ~0.6 * df_syll  (stiff)
    # flex = 120 -> ~1.8 * df_syll  (very bendy)
    df_scale = np.interp(f, [0, 120], [0.6, 1.8])
    df_raw = df_syll * df_scale

    # Cap around 50 so we don't go insane on long phrases.
    df_target = int(np.clip(df_raw, 12, 50))

    # Lambda schedule:
    # flex = 0   -> lambda high (stiff)
    # flex = 120 -> lambda ~0.005 (very permissive, follows local accents)
    lam_hi = 0.8    # stiffer, flatter global trend
    lam_lo = 0.005  # highly permissive, follows local accents
    lam = float(np.exp(np.interp(
        f,
        [0, 120],
        [np.log(lam_hi), np.log(lam_lo)]
    )))

    return df_target, lam


# ---------- Smoothing (LOWESS or GAM) ----------
def _smooth_curve(times, f0_values,
                  method="lowess",
                  lowess_frac=0.30,
                  gam_df=20, gam_degree=3, gam_lambda=0.6,
                  npoints=8000):
    """
    Smooth f0 values using LOWESS or GAM.

    Returns:
        time_smooth (np.ndarray): resampled time axis
        f0_smooth   (np.ndarray): smoothed f0 values
        f0_mean     (float): mean of the smoothed f0
        used_method (str): "LOWESS" or "GAM"

    GAM branch is statsmodels-compatible with 0.13.x and 0.14.x.
    We handle:
    - alpha passed to constructor in >=0.14
    - alpha passed to fit() in <=0.13
    - predict() signature differences
    """
    times = np.asarray(times, dtype=float)
    f0_values = np.asarray(f0_values, dtype=float)

    if times.size < 5:
        mean_val = float(np.nanmean(f0_values)) if f0_values.size else np.nan
        return times, f0_values, mean_val, "LOWESS"

    t_min, t_max = float(times.min()), float(times.max())
    span = max(t_max - t_min, 1e-9)
    t_smooth = np.linspace(t_min, t_max, int(min(max(npoints, 2000), 10000)))

    # --- GAM path (handles statsmodels 0.13 and 0.14 compat) ---
    if method.lower() == "gam" and _HAVE_GAM:
        try:
            t_norm = (times - t_min) / span
            t_grid = (t_smooth - t_min) / span

            Xs = t_norm.reshape(-1, 1)
            Xs_pred = t_grid.reshape(-1, 1)
            exog = np.ones((len(times), 1))
            exog_pred = np.ones((len(t_smooth), 1))

            # BSplines kw compat
            try:
                bs = BSplines(Xs, df=[int(gam_df)], degree=[int(gam_degree)], include_intercept=False)
            except TypeError:
                bs = BSplines(Xs, df=[int(gam_df)], degree=[int(gam_degree)])

            alpha = np.atleast_1d(float(gam_lambda))  # must be array-like

            # statsmodels 0.14+: alpha in constructor; 0.13: alpha in fit()
            try:
                gam = GLMGam(
                    f0_values,
                    exog=exog,
                    smoother=bs,
                    alpha=alpha,
                    family=sm.families.Gaussian()
                )
                res = gam.fit()
            except TypeError:
                gam = GLMGam(
                    f0_values,
                    exog=exog,
                    smoother=bs,
                    family=sm.families.Gaussian()
                )
                try:
                    res = gam.fit(alpha=alpha)
                except TypeError:
                    if _GAM_DEBUG:
                        print("[GAM warn] fit(alpha=...) not accepted; using model default penalty")
                    res = gam.fit()

            # predict API compat (kw vs positional)
            try:
                f0_smooth = np.asarray(
                    res.predict(exog=exog_pred, exog_smooth=Xs_pred),
                    dtype=float
                )
            except TypeError:
                f0_smooth = np.asarray(res.predict(exog_pred, Xs_pred), dtype=float)

            f0_mean = float(np.nanmean(f0_smooth))
            if _GAM_DEBUG:
                print(f"[GAM] df={int(gam_df)} deg={int(gam_degree)} λ={float(gam_lambda):.4f} → mean={f0_mean:.2f} Hz")
            return t_smooth, f0_smooth, f0_mean, "GAM"

        except Exception as e:
            if _GAM_DEBUG:
                print(f"[GAM→LOWESS fallback] {type(e).__name__}: {e}")

    # --- LOWESS (default and fallback) ---
    lowess_result = lowess(f0_values, times, frac=float(lowess_frac), return_sorted=True)
    tl = lowess_result[:, 0].astype(float)
    fl = lowess_result[:, 1].astype(float)

    try:
        spline = make_interp_spline(tl, fl, k=3)
        f0_smooth = spline(t_smooth)
        f0_mean = float(np.nanmean(f0_smooth))
        return t_smooth, f0_smooth, f0_mean, "LOWESS"
    except Exception:
        f0_mean = float(np.nanmean(fl))
        return tl, fl, f0_mean, "LOWESS"


def plot_f0_lowess(times, f0_values, centers_sr, sr_values, intensities,
                   cp_seq_num, cp_label, output_path, syll_intervals, xmin, xmax,
                   f0_min=50, f0_max=300, verse_text="", label_file="", tg=None,
                   lowess_frac=0.30,
                   smooth_method="lowess", gam_df=20, gam_degree=3, gam_lambda=0.6,
                   title_mode="auto",
                   gam_auto=False, gam_flex=75):
    """
    Plot a smoothed F0 contour (LOWESS or GAM) for a single CP.

    - Thickness encodes intensity (dB).
    - Color encodes a syllabic-rate proxy (SR) based on syllable duration.
    - Syllable boundaries and labels are shown on a lower panel.
    - Optional INT labels are overlaid if an INT tier is present.

    If gam_auto=True, gam_df and gam_lambda are overridden per CP using
    the syllable count and the Flex(%) slider (0..120).
    """

    if len(times) < 5:
        return

    # Auto-flex for GAM: choose df, lambda based on #syllables in this CP and flex slider
    if smooth_method == "gam" and gam_auto:
        n_syll, _, _ = _syllable_metrics(syll_intervals, xmin, xmax)
        auto_df, auto_lam = _gam_params_auto(n_syll, int(gam_flex))
        gam_df = auto_df
        gam_lambda = auto_lam

    # Compute smoothed curve (and know what actually ran)
    time_smooth, f0_smooth, f0_mean, used_method = _smooth_curve(
        times, f0_values,
        method=smooth_method,
        lowess_frac=lowess_frac,
        gam_df=gam_df, gam_degree=gam_degree, gam_lambda=gam_lambda
    )

    # Print one-liner into the UI status box
    if used_method == "LOWESS":
        print(f"[{label_file} CP{cp_seq_num}] smoother=LOWESS (frac={lowess_frac:.2f})")
    else:
        txt_auto = f" [AUTO flex={gam_flex}%]" if gam_auto else ""
        print(f"[{label_file} CP{cp_seq_num}] smoother=GAM (df={gam_df}, deg={gam_degree}, λ={gam_lambda:.4f}){txt_auto}")

    # Interpolations for line width and color (syllabic rate / intensity cues)
    sr_interp = np.interp(time_smooth, centers_sr, sr_values) if len(centers_sr) else np.zeros_like(time_smooth)
    int_interp = np.interp(time_smooth, times, intensities) if len(times) else np.zeros_like(time_smooth)

    # Normalizations for line styling
    def _norm(x):
        if len(x) == 0:
            return x
        mn, mx = np.min(x), np.max(x)
        return (x - mn) / (mx - mn + 1e-12)

    sr_norm = _norm(sr_interp)
    int_norm = _norm(int_interp)
    linewidths = 2.0 + 8.0 * int_norm

    # Syllabic metrics for legend
    n_syll, mean_ms, sr = _syllable_metrics(syll_intervals, xmin, xmax)

    # === Plot ===
    fig = plt.figure(figsize=(10, 5))
    gs = GridSpec(2, 1, height_ratios=[4, 1])
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1)

    # main F0 curve, colored by syllabic rate proxy (SR), thickness by intensity
    for i in range(1, len(time_smooth)):
        ax1.plot(
            time_smooth[i-1:i+1],
            f0_smooth[i-1:i+1],
            color=plt.cm.RdYlGn_r(sr_norm[i]),
            linewidth=linewidths[i]
        )

    leg_text = [
        f"mean f₀ = {f0_mean:.1f} Hz",
        f"mean syll = {mean_ms:.0f} ms" if not np.isnan(mean_ms) else "mean syll = n/a",
        f"SR = {sr:.2f} syll/s",
        f"syll = {n_syll:d}"
    ]
    ax1.axhline(f0_mean, color='gray', linestyle='--', linewidth=1, label=" | ".join(leg_text))
    ax1.legend(loc="upper right", fontsize=9)

    cp_title = build_cp_title(cp_seq_num, cp_label, mode=title_mode)
    title_str = f"{label_file} — {cp_title} — Verse: {verse_text}"
    ax1.set_title(title_str, fontsize=11)
    ax1.set_ylabel("Frequency (Hz)")
    ax1.set_xlim(xmin, xmax)
    ax1.set_ylim(f0_min, f0_max)

    ax2.set_xlim(xmin, xmax)
    ax2.set_ylim(0, 1)
    ax2.axis("off")

    # syllable boundaries + labels
    for s_start, s_end, s_lbl in _syllables_in_window(syll_intervals, xmin, xmax):
        ax1.axvline(s_start, color='gray', linestyle=':', linewidth=0.5)
        ax2.axvline(s_start, color='gray', linestyle=':', linewidth=0.5)
        ax2.text((s_start + s_end) / 2, 0.5, s_lbl, fontsize=10, ha='center', va='center')

    # INT tier labels (optional prosodic labels)
    if tg is not None and _tier_exists(tg, "INT"):
        int_tier = tg.getTier("INT")
        for start, end, int_label in int_tier.entries:
            if start >= xmin and end <= xmax:
                ax2.axvline(start, color='black', linestyle='--', linewidth=0.8)
                ax2.text(
                    (start + end) / 2, 0.2,
                    int_label, fontsize=10, color='gray',
                    ha='center', va='center'
                )

    plt.subplots_adjust(hspace=0.05)
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()


def process_cp_file(textgrid_path, wav_path, output_folder, label_file=None,
                    cp_indices=None, f0_min=50, f0_max=300, lowess_frac=0.30,
                    smooth_method="lowess", gam_df=20, gam_degree=3, gam_lambda=0.6,
                    title_mode="auto",
                    gam_auto=False, gam_flex=75):
    """
    Process a single TextGrid+WAV pair and generate CP-level contour plots.

    Args:
        textgrid_path (str): path to the TextGrid file
        wav_path (str): path to the WAV file
        output_folder (str): output directory for PNG plots
        label_file (str): short label used in plot titles
        cp_indices (list[int] or None): 1-based indices into the *valid* CP list
                                        (pauses removed). None → all valid CPs.
    """
    tg = textgrid.openTextgrid(textgrid_path, includeEmptyIntervals=False)

    if not _tier_exists(tg, "syll"):
        print(f"Tier 'syll' not found in {os.path.basename(textgrid_path)}")
        return

    syll_tier = tg.getTier("syll")
    syll_intervals = syll_tier.entries
    cp_valid = get_valid_cp_intervals(tg)  # filtered CPs

    if label_file is None:
        label_file = os.path.basename(textgrid_path).replace(".TextGrid", "")

    # Choose which CPs to process (based on valid indices)
    if cp_indices:
        chosen = [cp_valid[i-1] for i in cp_indices if 1 <= i <= len(cp_valid)]
        chosen_pairs = list(zip(cp_indices, chosen))
    else:
        chosen_pairs = list(zip(range(1, len(cp_valid)+1), cp_valid))

    for seq_num, (xmin, xmax, label) in chosen_pairs:
        voiced_segments = get_voiced_segments(tg, xmin, xmax)
        if not voiced_segments:
            continue

        times, f0_vals, intens_vals = extract_f0_intensity(wav_path, voiced_segments)
        times, f0_vals, intens_vals = filter_f0_outliers(
            times, f0_vals, intens_vals,
            z_threshold=2.5
        )
        if f0_vals.size == 0:
            continue

        # Syllables inside CP window
        syll_in_cp = [(s, e, l) for (s, e, l) in syll_intervals if s >= xmin and e <= xmax]
        centers_sr, sr_values = syllabic_rate_per_syllable(syll_in_cp)

        verse_text = get_current_verse_label(tg, xmin, xmax)
        out_name = f"{label_file}_CP{seq_num}.png"
        out_path = os.path.join(output_folder, out_name)

        # Let the plotting function handle smoothing (and print which one ran)
        plot_f0_lowess(
            times, f0_vals, centers_sr, sr_values, intens_vals,
            seq_num, label, out_path, syll_intervals, xmin, xmax,
            f0_min=f0_min, f0_max=f0_max, verse_text=verse_text,
            label_file=label_file, tg=tg,
            lowess_frac=lowess_frac,
            smooth_method=smooth_method, gam_df=gam_df, gam_degree=gam_degree,
            gam_lambda=gam_lambda,
            title_mode=title_mode,
            gam_auto=gam_auto, gam_flex=gam_flex
        )
        print(f"Saved: {out_name}")


def process_cp_folder_batch(folder, output_folder, f0_min=50, f0_max=300, lowess_frac=0.30,
                            smooth_method="lowess", gam_df=20, gam_degree=3, gam_lambda=0.6,
                            title_mode="auto",
                            gam_auto=False, gam_flex=75):
    """
    Batch process all TextGrid+WAV pairs in a folder.

    For each base name, the script looks for <base>.TextGrid and <base>.wav.
    """
    os.makedirs(output_folder, exist_ok=True)
    textgrid_files = [f for f in os.listdir(folder) if f.endswith(".TextGrid")]

    for tg_file in textgrid_files:
        base_name = tg_file.replace(".TextGrid", "")
        wav_file = base_name + ".wav"
        tg_path = os.path.join(folder, tg_file)
        wav_path = os.path.join(folder, wav_file)

        if os.path.exists(wav_path):
            print(f"Processing (CP): {base_name}")
            process_cp_file(
                tg_path, wav_path, output_folder,
                label_file=base_name,
                cp_indices=None,
                f0_min=f0_min, f0_max=f0_max,
                lowess_frac=lowess_frac,
                smooth_method=smooth_method,
                gam_df=gam_df, gam_degree=gam_degree, gam_lambda=gam_lambda,
                title_mode=title_mode,
                gam_auto=gam_auto, gam_flex=gam_flex
            )
        else:
            print(f"Missing WAV for {base_name}, skipping file.")


# ---------- UI Helpers ----------
def _scan_pairs(folder):
    """
    Scan a folder and return all base names that have both .TextGrid and .wav.
    """
    tg_files = sorted([f for f in os.listdir(folder) if f.endswith(".TextGrid")])
    items = []
    for tg in tg_files:
        base = tg[:-9]  # strip .TextGrid
        wav = base + ".wav"
        if os.path.exists(os.path.join(folder, wav)):
            items.append(base)
    return items

def _cp_options_for_file(folder, base):
    """
    Return a list of (label, index) pairs for valid CPs in a given file.
    """
    tg_path = os.path.join(folder, base + ".TextGrid")
    tg = textgrid.openTextgrid(tg_path, includeEmptyIntervals=False)
    cps = get_valid_cp_intervals(tg)  # only valid CPs
    options = []
    for i, (s, e, lab) in enumerate(cps, start=1):
        options.append((f"CP{i} [{s:.2f}-{e:.2f}] {lab}", i))
    return options


# ---------- UI (revamp + smoother selector + updated Auto Flex) ----------
def launch_ui(default_input="/Users/Federico/Desktop/materiali_tesi/Montale/prova_meriggiare/",
              default_output="/Users/Federico/Desktop/materiali_tesi/Montale/prova_meriggiare/",
              default_f0_min=50, default_f0_max=300, default_frac=0.22):
    """
    Launch an interactive Jupyter UI for CP-level intonation visualization.

    If ipywidgets is not available, a batch run is executed instead.
    """

    if not _HAVE_WIDGETS:
        print("Interactive UI not available (ipywidgets missing). Running batch mode.")
        process_cp_folder_batch(
            default_input, default_output,
            f0_min=default_f0_min,
            f0_max=default_f0_max,
            lowess_frac=default_frac,
            smooth_method="lowess",
            title_mode="auto"
        )
        return

    UI_MAX_WIDTH = "980px"

    # Small CSS polish (card layout, light background, subtle shadows)
    display(HTML("""
    <style>
      .cpviz-root {
        max-width: 980px;
        margin: 0 auto;
        padding: 8px 4px 16px 4px;
        background: #f7f7fb;
      }
      .cpviz-h1 {
        font-size: 20px;
        font-weight: 700;
        margin: 4px 0 0 0;
        font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      }
      .cpviz-sub {
        font-size: 12px;
        color: #666;
        margin: 2px 0 12px 0;
        font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      }
      .cpviz-hr  {
        border:none;
        border-top:1px solid #e5e5e5;
        margin: 8px 0;
      }
      .cpviz-card {
        border:1px solid #ddd;
        border-radius:10px;
        padding:12px 14px;
        margin:10px 0;
        background: #ffffff;
        box-shadow: 0 1px 3px rgba(0,0,0,0.04);
      }
      .cpviz-card h3 {
        margin:0 0 8px 0;
        font-size:14px;
        font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      }
      .cpviz-help, .cpviz-note {
        color:#666;
        font-size:12px;
        line-height:1.4;
        font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      }
      .cpviz-summary {
        font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
        font-size:12px;
        white-space:pre-wrap;
      }
      .cpviz-field {
        display:flex;
        flex-direction:column;
        gap:4px;
      }
      .cpviz-label {
        font-size:12px;
        font-weight:600;
        color:#333;
        font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      }
      .widget-button {
        font-size: 12px;
      }
    </style>
    """))

    # --- Header ---
    header = W.HTML(
        "<div class='cpviz-root'>"
        "<div class='cpviz-h1'>CP Intonation Visualizer</div>"
        f"<div class='cpviz-sub'>Interactive UI • LOWESS/GAM smoothing • Auto Flex GAM (df≈50 λ≈0.005 at high flex) • statsmodels: {_SM_VER}</div>"
        "</div>"
    )
    display(header)

    # --- Card builder ---
    def card(title, body_widget, extra_top_html=""):
        head = W.HTML(f"<h3>{title}</h3>{extra_top_html}")
        box = W.VBox([head, body_widget], layout=W.Layout(width="100%"))
        box.add_class("cpviz-card")
        return box

    # --- Mode & Paths ---
    mode = W.ToggleButtons(
        options=[("Batch (all CP)", "batch"),
                 ("Interactive (pick files/CP)", "interactive")],
        value="interactive", description="Mode:", style={"button_width": "260px"}
    )
    in_dir  = W.Text(value=default_input, description="Input dir:", layout=W.Layout(width="100%"))
    out_dir = W.Text(value=default_output, description="Output dir:", layout=W.Layout(width="100%"))
    scan_btn = W.Button(description="Scan", icon="search")
    reset_paths = W.Button(description="Reset", icon="refresh")
    files_filter = W.Text(value="", description="Filter:", placeholder="part of filename…", layout=W.Layout(width="100%"))
    files_select = W.SelectMultiple(options=[], description="Files:", rows=12, layout=W.Layout(width="100%"))
    files_select.layout = W.Layout(width="100%", max_height="320px", overflow_y="auto")
    sel_all = W.Button(description="Select all", icon="check")
    sel_none = W.Button(description="None", icon="ban")

    # --- Preset & Plot Settings ---
    smoother = W.Dropdown(
        options=[("LOWESS", "lowess")] + (
            [("GAM (statsmodels)", "gam")] if _HAVE_GAM else
            [("GAM (unavailable)", "lowess")]
        ),
        value="lowess", description=""
    )

    preset = W.Dropdown(options=list(PRESETS.keys()), value="Custom", description="")
    f0_min = W.IntText(value=default_f0_min, description="")
    f0_max = W.IntText(value=default_f0_max, description="")
    title_mode = W.Dropdown(
        options=[("Auto (no dup)", "auto"), ("Number only", "number"),
                 ("Label only", "label"), ("Number + Label", "both")],
        value="auto", description=""
    )

    # LOWESS smoothing as numeric field
    frac = W.BoundedFloatText(value=default_frac, min=0.05, max=0.80, step=0.01, description="")

    # GAM parameters (manual mode defaults)
    gam_df = W.BoundedIntText(value=36, min=6, max=80, step=1, description="")
    gam_degree = W.BoundedIntText(value=3, min=2, max=5, step=1, description="")
    gam_lambda = W.BoundedFloatText(value=0.15, min=0.0, max=100.0, step=0.05, description="")

    # Auto Flex controls
    gam_auto = W.Checkbox(value=True, description="GAM Auto Flex")
    gam_flex = W.IntSlider(value=90, min=0, max=120, step=5, description="Flex (%)")

    def _field(label_text, widget, note_text=None):
        label = W.HTML(f"<div class='cpviz-label'>{label_text}</div>")
        kids = [label, widget]
        if note_text:
            kids.append(W.HTML(f"<div class='cpviz-note'>{note_text}</div>"))
        box = W.VBox(kids, layout=W.Layout(width="100%"))
        box.add_class("cpviz-field")
        return box

    f_smoother  = _field("Smoother", smoother, ("GAM unavailable → LOWESS will be used." if not _HAVE_GAM else None))
    f_preset    = _field("Preset", preset)
    f_fmin      = _field("f₀ min (Hz)", f0_min)
    f_fmax      = _field("f₀ max (Hz)", f0_max)
    f_title     = _field("Title mode", title_mode)
    f_frac      = _field("LOWESS frac", frac, "0.05–0.80 (higher = flatter)")

    f_gdf       = _field("GAM df (splines)", gam_df, "Higher df = more bends (detail)")
    f_gdeg      = _field("GAM degree", gam_degree, "3 = cubic, smooth turns")
    f_glam      = _field("GAM λ (penalty)", gam_lambda, "Lower λ = more wiggle")

    f_gam_auto  = _field("GAM Auto Flex", gam_auto, "Learn df / λ from syllables")
    f_gam_flex  = _field("Flex (%)", gam_flex, "0=flat trend, 120=expressive (~df50 λ0.005)")

    preset_grid = W.GridBox(
        children=[
            f_smoother, f_preset, f_fmin, f_fmax, f_title,
            f_frac,         # LOWESS control
            f_gdf, f_gdeg, f_glam,    # manual GAM tuning
            f_gam_auto, f_gam_flex    # auto flex tuning
        ],
        layout=W.Layout(
            width="100%",
            grid_template_columns="repeat(auto-fit, minmax(180px, 1fr))",
            grid_gap="12px"
        )
    )

    def _toggle_smoother_fields(*_):
        if smoother.value == "gam" and _HAVE_GAM:
            # Hide LOWESS control
            f_frac.layout.display = "none"

            # Always show Auto Flex controls in GAM mode
            f_gam_auto.layout.display = ""
            f_gam_flex.layout.display = ""

            # Manual GAM controls only if Auto Flex is OFF
            if gam_auto.value:
                f_gdf.layout.display = "none"
                f_gdeg.layout.display = "none"
                f_glam.layout.display = "none"
            else:
                f_gdf.layout.display = ""
                f_gdeg.layout.display = ""
                f_glam.layout.display = ""
        else:
            # LOWESS visible
            f_frac.layout.display = ""

            # Hide GAM fields
            f_gdf.layout.display = "none"
            f_gdeg.layout.display = "none"
            f_glam.layout.display = "none"
            f_gam_auto.layout.display = "none"
            f_gam_flex.layout.display = "none"

    _toggle_smoother_fields()
    smoother.observe(_toggle_smoother_fields, names="value")
    gam_auto.observe(_toggle_smoother_fields, names="value")

    preset_help = W.HTML(
        "<div class='cpviz-note'>"
        "Presets prefill f₀ bounds and LOWESS frac. "
        "LOWESS = locally weighted regression with frac. "
        "GAM = penalized regression spline. "
        "Auto Flex adapts df and λ per CP; high Flex (~120) targets df≈50 and λ≈0.005 "
        "to match detailed LOWESS-like melody."
        "</div>"
    )
    preset_card = card("2) Presets & Plot Settings", W.VBox([preset_grid, preset_help]))

    # --- CP selection card ---
    cp_accordion = W.Accordion(children=[], selected_index=None)
    cp_label = W.HTML("<b>CP selection per file</b> <span class='cpviz-help'>(empty = all valid CPs)</span>")
    selection_card = card("3) CP Selection", W.VBox([cp_label, cp_accordion]))

    # --- Summary & Run card ---
    summary = W.HTML("<div class='cpviz-summary'>—</div>")
    run_btn = W.Button(description="Run", icon="play", button_style="success")
    progress = W.IntProgress(value=0, min=0, max=100, description="Progress:", bar_style="info",
                             layout=W.Layout(width="100%"))
    status = W.Output()
    run_card = card("4) Summary & Run",
                    W.VBox([
                        W.HTML("<b>Current settings summary</b>"),
                        summary,
                        progress,
                        W.HBox([run_btn], layout=W.Layout(justify_content="flex-end")),
                        W.HTML("<div class='cpviz-hr'></div>"),
                        status
                    ]),
                    extra_top_html="<div class='cpviz-help'>Review parameters, then run interactive or batch processing.</div>")

    # ---------- Paths card assembly ----------
    paths_grid = W.GridBox(
        children=[in_dir, out_dir],
        layout=W.Layout(
            width="100%",
            grid_template_columns="repeat(auto-fit, minmax(260px, 1fr))",
            grid_gap="12px"
        )
    )
    path_btns = W.HBox([scan_btn, reset_paths], layout=W.Layout(justify_content="flex-start"))
    files_controls = W.GridBox(
        children=[files_filter, files_select, sel_all, sel_none],
        layout=W.Layout(
            width="100%",
            grid_template_columns="repeat(auto-fit, minmax(220px, 1fr))",
            grid_gap="12px",
            align_items="flex-start"
        )
    )
    paths_card = card(
        "1) Mode & Paths",
        W.VBox([
            mode,
            W.HTML("<div class='cpviz-hr'></div>"),
            paths_grid, path_btns,
            W.HTML("<div class='cpviz-hr'></div>"),
            W.HTML("<b>Available files</b>"),
            files_controls
        ]),
        extra_top_html="<div class='cpviz-help'>Set folders and scan to find TextGrid+WAV pairs.</div>"
    )

    # ---------- Behaviors ----------
    def _apply_preset(change=None):
        p = PRESETS.get(preset.value, {})
        if p.get("fmin") is not None:
            f0_min.value = p["fmin"]
        if p.get("fmax") is not None:
            f0_max.value = p["fmax"]
        if p.get("frac") is not None:
            frac.value = p["frac"]
        _update_summary()

    def _populate_files(_=None):
        with status:
            clear_output()
            if not os.path.isdir(in_dir.value):
                print("Input folder does not exist.")
                files_select.options = []
                return
            items = _scan_pairs(in_dir.value)
            fil = files_filter.value.strip().lower()
            if fil:
                items = [b for b in items if fil in b.lower()]
            files_select.options = items
            print(f"Found {len(items)} TextGrid+WAV pairs.")
        _populate_cps()
        _update_summary()

    def _reset_paths(_=None):
        in_dir.value = default_input
        out_dir.value = default_output
        files_filter.value = ""
        files_select.options = []
        cp_accordion.children = []
        _update_summary()

    def _populate_cps(change=None):
        children = []
        titles = []
        for base in files_select.value:
            try:
                opts = _cp_options_for_file(in_dir.value, base)
            except Exception:
                opts = []
            multisel = W.SelectMultiple(options=opts, rows=min(10, max(5, len(opts))))
            children.append(multisel)
            titles.append(base)
        cp_accordion.children = children
        for i, t in enumerate(titles):
            cp_accordion.set_title(i, t)
        _update_summary()

    def _files_filter_change(change=None):
        _populate_files()

    def _select_all(_=None):
        files_select.value = tuple(files_select.options)
        _populate_cps()

    def _select_none(_=None):
        files_select.value = tuple()
        _populate_cps()

    def _update_summary(_=None):
        mode_s = "Batch" if mode.value == "batch" else "Interactive"
        chosen_files = list(files_select.value)
        fmin = int(f0_min.value)
        fmax = int(f0_max.value)
        tmode = title_mode.value

        if smoother.value == "gam" and _HAVE_GAM:
            if gam_auto.value:
                sm_descr = f"GAM Auto Flex (flex={int(gam_flex.value)}%, df≈≤50, λ→0.005 at high flex)"
            else:
                sm_descr = (f"GAM Manual (df={gam_df.value}, "
                            f"deg={gam_degree.value}, λ={gam_lambda.value})")
        else:
            sm_descr = (
                f"LOWESS (frac={frac.value:.2f})"
                + ("" if _HAVE_GAM or smoother.value != "gam"
                   else " [GAM unavailable → using LOWESS]")
            )

        lines = [
            f"Mode: {mode_s}",
            f"Input:  {in_dir.value}",
            f"Output: {out_dir.value}",
            f"Preset: {preset.value} | f₀ [{fmin}, {fmax}] Hz",
            f"Smoother: {sm_descr} | Title mode: {tmode}",
            f"GAM available: {'yes' if _HAVE_GAM else 'no'} (statsmodels {_SM_VER})"
        ]
        if mode.value == "interactive":
            lines.append(f"Selected files: {len(chosen_files)}")
            for i, base in enumerate(chosen_files, start=1):
                try:
                    opts = _cp_options_for_file(in_dir.value, base)
                    n_tot = len(opts)
                except Exception:
                    n_tot = 0
                try:
                    w_child = cp_accordion.children[i-1]
                    n_sel = len(w_child.value) if w_child.value else n_tot
                except Exception:
                    n_sel = n_tot
                lines.append(f"  - {base}: {n_sel}/{n_tot} CP")
        summary.value = "<div class='cpviz-summary'>" + "\n".join(lines) + "</div>"

    def _run(_=None):
        with status:
            clear_output()
            os.makedirs(out_dir.value, exist_ok=True)

            fmin = int(f0_min.value)
            fmax = int(f0_max.value)
            tmode = title_mode.value

            # Decide smoothing method + params
            sm_method = smoother.value if _HAVE_GAM else "lowess"
            low_frac = float(frac.value)

            g_df = int(gam_df.value)
            g_deg = int(gam_degree.value)
            g_lam = float(gam_lambda.value)

            g_auto_flag = (smoother.value == 'gam' and _HAVE_GAM and gam_auto.value)
            g_flex_val = int(gam_flex.value)

            if fmin >= fmax:
                print("f₀ min must be less than f₀ max.")
                return

            progress.value = 0
            progress.bar_style = "info"

            if mode.value == "batch":
                all_pairs = _scan_pairs(in_dir.value)
                todo = len(all_pairs)
                progress.max = max(1, todo)
                print(f"Batch — f₀ [{fmin}, {fmax}] Hz ; smoother={sm_method.upper()} ; title={tmode}")
                for i, base in enumerate(all_pairs, start=1):
                    tg_path = os.path.join(in_dir.value, base + ".TextGrid")
                    wav_path = os.path.join(in_dir.value, base + ".wav")
                    if not os.path.exists(wav_path):
                        print(f"Missing WAV for {base}, skipping.")
                    else:
                        print(f"Processing (CP): {base}")
                        process_cp_file(
                            tg_path, wav_path, out_dir.value,
                            label_file=base, cp_indices=None,
                            f0_min=fmin, f0_max=fmax,
                            lowess_frac=low_frac,
                            smooth_method=sm_method,
                            gam_df=g_df, gam_degree=g_deg, gam_lambda=g_lam,
                            title_mode=tmode,
                            gam_auto=g_auto_flag, gam_flex=g_flex_val
                        )
                    progress.value = i
                progress.bar_style = "success"
                print("Batch completed.")
                return

            # interactive mode
            if not files_select.value:
                print("No file selected.")
                return

            chosen_files = list(files_select.value)
            progress.max = len(chosen_files)
            print(f"Interactive — f₀ [{fmin}, {fmax}] Hz ; smoother={sm_method.upper()} ; title={tmode}")

            for idx, base in enumerate(chosen_files):
                tg_path = os.path.join(in_dir.value, base + ".TextGrid")
                wav_path = os.path.join(in_dir.value, base + ".wav")
                if not os.path.exists(wav_path):
                    print(f"Missing WAV for {base}, skipping.")
                    progress.value = idx + 1
                    continue

                sel_widget = cp_accordion.children[idx] if idx < len(cp_accordion.children) else None
                if sel_widget and sel_widget.value:
                    cp_idx = sorted(list(sel_widget.value))  # 1-based on valid CPs
                    print(f"{base}: selected CP {cp_idx}")
                else:
                    cp_idx = None
                    print(f"{base}: all valid CPs")

                process_cp_file(
                    tg_path, wav_path, out_dir.value,
                    label_file=base, cp_indices=cp_idx,
                    f0_min=fmin, f0_max=fmax,
                    lowess_frac=low_frac,
                    smooth_method=sm_method,
                    gam_df=g_df, gam_degree=g_deg, gam_lambda=g_lam,
                    title_mode=tmode,
                    gam_auto=g_auto_flag, gam_flex=g_flex_val
                )
                progress.value = idx + 1

            progress.bar_style = "success"
            print("Interactive processing finished.")

    # Wiring
    preset.observe(_apply_preset, names="value")
    for w in (
        f0_min, f0_max, title_mode, smoother, frac,
        gam_df, gam_degree, gam_lambda,
        gam_auto, gam_flex,
        mode
    ):
        w.observe(_update_summary, names="value")
    scan_btn.on_click(_populate_files)
    reset_paths.on_click(_reset_paths)
    files_filter.observe(_files_filter_change, names="value")
    files_select.observe(_populate_cps, names="value")
    sel_all.on_click(_select_all)
    sel_none.on_click(_select_none)
    run_btn.on_click(_run)

    # Final display in a capped container
    container = W.VBox([
        paths_card,
        preset_card,
        selection_card,
        run_card
    ], layout=W.Layout(width="100%", max_width=UI_MAX_WIDTH, margin="0 auto"))
    display(container)


# === ENTRYPOINT ===
if __name__ == "__main__":
    from IPython import get_ipython
    try:
        in_nb = get_ipython() is not None
        if in_nb and _HAVE_WIDGETS:
            launch_ui()  # Launch UI in notebooks
        elif in_nb and not _HAVE_WIDGETS:
            print("ipywidgets not available in this kernel. Install and restart:")
            print("  pip install ipywidgets")
        else:
            print("No IPython detected → running batch.")
            input_dir = "/Users/Federico/Desktop/materiali_tesi/Montale/prova_meriggiare/"
            output_dir = "/Users/Federico/Desktop/materiali_tesi/Montale/prova_meriggiare/"
            process_cp_folder_batch(
                input_dir, output_dir,
                f0_min=50, f0_max=300,
                lowess_frac=0.22,
                smooth_method='lowess',
                title_mode="auto",
                gam_auto=False, gam_flex=90
            )
    except Exception:
        import traceback; traceback.print_exc()
        print("UI crashed; falling back to batch.")
        input_dir = "/Users/Federico/Desktop/materiali_tesi/Montale/prova_meriggiare/"
        output_dir = "/Users/Federico/Desktop/materiali_tesi/Montale/prova_meriggiare/"
        process_cp_folder_batch(
            input_dir, output_dir,
            f0_min=50, f0_max=300,
            lowess_frac=0.22,
            smooth_method='lowess',
            title_mode="auto",
            gam_auto=False, gam_flex=90
        )

HTML(value="<div class='cpviz-root'><div class='cpviz-h1'>CP Intonation Visualizer</div><div class='cpviz-sub'…

In [ ]:
# ============================================================
# EXTRA CELL — Export CP-level prosodic metrics to CSV
# Uses the functions already defined in the visualizer notebook
# ============================================================

import os
import numpy as np
import pandas as pd
from praatio import textgrid

def _hz_to_st_span(f0_min, f0_max):
    """
    Convert an F0 range from Hz to semitones.
    Returns NaN if the input values are invalid.
    """
    if f0_min is None or f0_max is None:
        return np.nan
    if f0_min <= 0 or f0_max <= 0:
        return np.nan
    return 12.0 * np.log2(f0_max / f0_min)

def _safe_mean(x):
    """
    Safe mean for numeric arrays.
    """
    x = np.asarray(x, dtype=float)
    return float(np.nanmean(x)) if x.size else np.nan

def _safe_std(x):
    """
    Safe standard deviation for numeric arrays.
    """
    x = np.asarray(x, dtype=float)
    return float(np.nanstd(x)) if x.size else np.nan

def _safe_min(x):
    """
    Safe minimum for numeric arrays.
    """
    x = np.asarray(x, dtype=float)
    return float(np.nanmin(x)) if x.size else np.nan

def _safe_max(x):
    """
    Safe maximum for numeric arrays.
    """
    x = np.asarray(x, dtype=float)
    return float(np.nanmax(x)) if x.size else np.nan

def _syllable_durations_in_window(syll_intervals, xmin, xmax):
    """
    Return syllable durations (in seconds) for all valid syllables
    fully contained in the analysis window.
    """
    kept = _syllables_in_window(syll_intervals, xmin, xmax)
    return np.array([(e - s) for s, e, _ in kept], dtype=float)

def extract_cp_metrics_file(
    textgrid_path,
    wav_path,
    label_file=None,
    cp_indices=None,              # 1-based indices in the valid-CP list
    lowess_frac=0.30,
    smooth_method="lowess",
    gam_df=20,
    gam_degree=3,
    gam_lambda=0.6,
    gam_auto=False,
    gam_flex=75
):
    """
    Extract CP-level prosodic metrics from one TextGrid+WAV pair.

    Parameters
    ----------
    textgrid_path : str
        Path to the TextGrid file.
    wav_path : str
        Path to the WAV file.
    label_file : str or None
        File label to store in the output table. If None, it is inferred
        from the TextGrid file name.
    cp_indices : list[int] or None
        Optional list of 1-based indices over valid CPs only
        (pause/noise CPs excluded). If None, all valid CPs are processed.
    lowess_frac, smooth_method, gam_df, gam_degree, gam_lambda, gam_auto, gam_flex
        Same smoothing settings used in the plotting notebook.

    Returns
    -------
    rows : list[dict]
        One dictionary per CP.
    """
    tg = textgrid.openTextgrid(textgrid_path, includeEmptyIntervals=False)

    if not _tier_exists(tg, "syll"):
        print(f"Tier 'syll' not found in {os.path.basename(textgrid_path)}")
        return []

    if label_file is None:
        label_file = os.path.basename(textgrid_path).replace(".TextGrid", "")

    syll_tier = tg.getTier("syll")
    syll_intervals = syll_tier.entries
    cp_valid = get_valid_cp_intervals(tg)

    if cp_indices:
        chosen = [cp_valid[i - 1] for i in cp_indices if 1 <= i <= len(cp_valid)]
        chosen_pairs = list(zip(cp_indices, chosen))
    else:
        chosen_pairs = list(zip(range(1, len(cp_valid) + 1), cp_valid))

    rows = []

    for seq_num, (xmin, xmax, cp_label) in chosen_pairs:
        cp_duration = float(xmax - xmin)
        if cp_duration <= 0:
            continue

        # Get voiced phonetic segments inside the CP
        voiced_segments = get_voiced_segments(tg, xmin, xmax)
        voiced_duration = float(sum(e - s for s, e in voiced_segments)) if voiced_segments else 0.0
        voiced_ratio = voiced_duration / cp_duration if cp_duration > 0 else np.nan

        # Extract raw F0 and intensity values, then apply the same outlier filter
        times, f0_vals, intens_vals = extract_f0_intensity(wav_path, voiced_segments)
        times, f0_vals, intens_vals = filter_f0_outliers(
            times, f0_vals, intens_vals, z_threshold=2.5
        )

        # Compute syllable-based metrics
        n_syll, mean_syll_ms, syll_rate = _syllable_metrics(syll_intervals, xmin, xmax)
        syll_durs = _syllable_durations_in_window(syll_intervals, xmin, xmax)

        if syll_durs.size and np.nanmean(syll_durs) > 0:
            cv_syllable_duration = float(np.nanstd(syll_durs) / np.nanmean(syll_durs))
        else:
            cv_syllable_duration = np.nan

        # Raw F0 metrics
        f0_sd_raw = _safe_std(f0_vals)
        f0_min_raw = _safe_min(f0_vals)
        f0_max_raw = _safe_max(f0_vals)
        f0_range_st_raw = _hz_to_st_span(f0_min_raw, f0_max_raw)

        # Raw intensity metrics
        intensity_mean_db = _safe_mean(intens_vals)
        intensity_sd_db = _safe_std(intens_vals)

        # Smoothed F0 metrics, aligned with the visualizer settings
        f0_mean_hz_smooth = np.nan
        f0_slope_hz_per_s_smooth = np.nan

        if len(times) >= 5:
            local_gam_df = gam_df
            local_gam_lambda = gam_lambda

            # If GAM auto-flex is enabled, derive df and lambda from
            # the number of syllables in the current CP
            if smooth_method == "gam" and gam_auto:
                auto_df, auto_lam = _gam_params_auto(n_syll, int(gam_flex))
                local_gam_df = auto_df
                local_gam_lambda = auto_lam

            t_smooth, f0_smooth, f0_mean_hz_smooth, used_method = _smooth_curve(
                times,
                f0_vals,
                method=smooth_method,
                lowess_frac=lowess_frac,
                gam_df=local_gam_df,
                gam_degree=gam_degree,
                gam_lambda=local_gam_lambda
            )

            # Estimate the global slope of the smoothed contour
            if len(t_smooth) >= 2 and len(f0_smooth) >= 2:
                smooth_dur = float(t_smooth[-1] - t_smooth[0])
                if smooth_dur > 0:
                    f0_slope_hz_per_s_smooth = float((f0_smooth[-1] - f0_smooth[0]) / smooth_dur)

        row = {
            "file_id": label_file,
            "cp_index": seq_num,
            "cp_label": cp_label,
            "cp_start_s": float(xmin),
            "cp_end_s": float(xmax),

            # Core duration / rhythm metrics
            "cp_duration_s": cp_duration,
            "n_syll": int(n_syll),
            "syll_rate": syll_rate,
            "mean_syllable_duration_ms": mean_syll_ms,

            # Smoothed melodic central tendency
            "f0_mean_hz_smooth": float(f0_mean_hz_smooth) if not np.isnan(f0_mean_hz_smooth) else np.nan,

            # Intensity central tendency
            "intensity_mean_db": intensity_mean_db,

            # Comparative metrics across CPs
            "f0_sd_hz_raw": f0_sd_raw,
            "f0_range_st_raw": f0_range_st_raw,
            "intensity_sd_db": intensity_sd_db,
            "voiced_ratio": voiced_ratio,

            # More interpretive metrics
            "f0_slope_hz_per_s_smooth": f0_slope_hz_per_s_smooth,
            "cv_syllable_duration": cv_syllable_duration
        }

        rows.append(row)

    return rows

def export_cp_metrics_csv(
    folder,
    output_csv,
    selected_files=None,          # list of basenames, or None = all paired files
    selected_cp_map=None,         # dict: {basename: [1,3,4], ...}
    lowess_frac=0.30,
    smooth_method="lowess",
    gam_df=20,
    gam_degree=3,
    gam_lambda=0.6,
    gam_auto=False,
    gam_flex=75
):
    """
    Export CP-level prosodic metrics to a CSV file.

    Parameters
    ----------
    folder : str
        Folder containing matched .TextGrid and .wav files.
    output_csv : str
        Output CSV path.
    selected_files : list[str] or None
        Optional list of basenames to process. If None, all available pairs
        in the folder are processed.
    selected_cp_map : dict or None
        Optional mapping from basename to a list of 1-based valid CP indices.
        Example:
            {
                "file_A": [1, 2, 5],
                "file_B": [3]
            }
        If a file is not present in the dictionary, all valid CPs are exported.
    lowess_frac, smooth_method, gam_df, gam_degree, gam_lambda, gam_auto, gam_flex
        Same smoothing settings used in the plotting notebook.

    Returns
    -------
    df : pandas.DataFrame
        Output table saved to CSV.
    """
    if selected_cp_map is None:
        selected_cp_map = {}

    all_bases = _scan_pairs(folder)
    if selected_files is None:
        bases = all_bases
    else:
        bases = [b for b in selected_files if b in all_bases]

    all_rows = []

    for base in bases:
        tg_path = os.path.join(folder, base + ".TextGrid")
        wav_path = os.path.join(folder, base + ".wav")

        if not os.path.exists(wav_path):
            print(f"Missing WAV for {base}, skipping.")
            continue

        cp_indices = selected_cp_map.get(base, None)

        rows = extract_cp_metrics_file(
            textgrid_path=tg_path,
            wav_path=wav_path,
            label_file=base,
            cp_indices=cp_indices,
            lowess_frac=lowess_frac,
            smooth_method=smooth_method,
            gam_df=gam_df,
            gam_degree=gam_degree,
            gam_lambda=gam_lambda,
            gam_auto=gam_auto,
            gam_flex=gam_flex
        )
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"Saved CSV: {output_csv}")
    print(f"Rows exported: {len(df)}")
    return df

In [ ]:
df_cp = export_cp_metrics_csv(
    folder="/Users/Federico/Desktop/materiali_tesi/Montale/prova_meriggiare/",
    output_csv="/Users/Federico/Desktop/materiali_tesi/Montale/prova_meriggiare/cp_metrics_selected.csv",
    smooth_method="gam",   # or "lowess"
    gam_auto=True,
    gam_flex=90
)

df_cp.head()

In [ ]:
df_cp = export_cp_metrics_csv(
    folder="/Users/Federico/Desktop/materiali_tesi/Montale/prova_meriggiare/",
    output_csv="/Users/Federico/Desktop/materiali_tesi/Montale/prova_meriggiare/cp_metrics_selected.csv",
    selected_files=[
        "bianchi_meriggiare_tidied",
        "rosselli_se_sinistramente"
    ],
    selected_cp_map={
        "bianchi_meriggiare_tidied": [1, 2, 4],
        "rosselli_se_sinistramente": [3]
    },
    smooth_method="gam",
    gam_auto=True,
    gam_flex=90
)

df_cp.head()